# Antisense NOT gate

Manual test harness for `engine.gates.antisense.AntisenseNotGate`. This family is
**planned, not implemented** (`available = False`): `is_compatible()` returns a
`Compatibility.no(...)` rather than raising, and the rest print `pending Step 5`.

## The mechanism

An antisense RNA complementary to the payload's RBS and start region pairs with it
and blocks translation. Expression is **ON by default** and switched **OFF** when
the trigger appears — the inverse of a toehold, and what makes `NOT` buildable so a
circuit can use a down-regulated gene as an input.

Note the safety inversion: a *failed* antisense gate expresses the payload rather
than staying dark. And `predicted_leakage` here means residual expression when the
trigger **is** present — same metric name, opposite biological event.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
host = fx.Host.ECOLI          # ECOLI | YEAST | HUMAN
gate = fx.antisense(host=host)
fx.describe_gate(gate)        # available = False

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
# A NOT gates on a transcript that must be ABSENT, so the trigger is a repressor.
triggers = fx.sample_trigger_set(n_activators=0, n_repressors=1)
constraints = fx.sample_constraints()

for t in triggers.repressors:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()` — returns `Compatibility.no(...)` while planned

Not a raise: the stage must skip a planned family cleanly. Once implemented,
Step 5 adds the real checks (single input, duplex-length minimum, trigger
accessibility).

In [ ]:
fx.attempt('is_compatible', lambda: gate.is_compatible(triggers, constraints))

## `generate_designs()` *(Step 5)*

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()` *(Step 5)*

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [ ]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.antisense(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```